In [1]:
from nnsight import LanguageModel
from nnsight.models.UnifiedTransformer import UnifiedTransformer

import sys

sys.path.append("../../../")

from nngine import alter

import torch 
import einops

model = LanguageModel("openai-community/gpt2", device_map="auto", dispatch=True)
alter(model)

tokenizer = model.tokenizer

tl_model = UnifiedTransformer(
    'gpt2-small',
    processing=False,
    center_writing_weights=False,
    center_unembed=False,
    fold_ln=False,
    device="cuda",
)

tl_model.set_use_hook_mlp_in(True)
tl_model.set_use_split_qkv_input(True)
tl_model.set_use_attn_result(True)

def test_resolution(ground_truth, sample):
    for resolution in [1e-12, 1e-10, 1e-6, 1e-4, 1e-3, 1e-2, 1.0]:
        pct = (sample - ground_truth > resolution).float().mean().item()
        print(f'pct out of range {resolution:.1e} = {pct:.2%}')

Loaded pretrained model gpt2-small into HookedTransformer


In [2]:
sample = tokenizer.encode("Susan and Mary went to the store, Susan went in and")

with model.trace(sample):

    test = model.transformer.layers[0].attn.split_q.input.grad.save()

    logits = model.output.logits[:,-1,:]
    value = logits.sum()
    value.backward()

with tl_model.trace(sample):
    tl_test = tl_model.blocks[0].hook_q_input.input[0][0].grad.save()

    logits = tl_model.unembed.output[:,-1,:]
    value = logits.sum()
    value.backward()

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [3]:
test_resolution(test, tl_test)

pct out of range 1.0e-12 = 45.63%
pct out of range 1.0e-10 = 45.63%
pct out of range 1.0e-06 = 45.54%
pct out of range 1.0e-04 = 40.45%
pct out of range 1.0e-03 = 21.03%
pct out of range 1.0e-02 = 1.32%
pct out of range 1.0e+00 = 0.00%


In [9]:
with model.trace(sample):

    test = model.transformer.layers[0].attn.attn_result.output.save()

with tl_model.trace(sample):
    tl_test = tl_model.blocks[0].attn.hook_result.output.save()


In [11]:
print(test.shape)
print(tl_test.shape)

test_resolution(tl_test,test)

torch.Size([1, 12, 12, 768])
torch.Size([1, 12, 12, 768])
pct out of range 1.0e-12 = 44.73%
pct out of range 1.0e-10 = 44.72%
pct out of range 1.0e-06 = 0.01%
pct out of range 1.0e-04 = 0.00%
pct out of range 1.0e-03 = 0.00%
pct out of range 1.0e-02 = 0.00%
pct out of range 1.0e+00 = 0.00%


# TESTING ATTENTION RESULT (GOOD)

In [ ]:
sample = tokenizer.encode("Susan and Mary went to the store, Susan went in and")

with model.trace(sample):
    c_proj = model.transformer.h[8].attn.c_proj

    heads = einops.rearrange(
        c_proj.input[0][0], 
        "batch seq (head_idx head_dim) -> batch seq head_idx head_dim", 
        head_idx=12, 
        head_dim=64
    )

    w_o_split = einops.rearrange(
        c_proj.weight,
        "(head_idx head_dim) d_model \
            -> head_idx head_dim d_model",
        head_idx=12,
        head_dim=64
    )

    attn_out = einops.einsum(
        heads, w_o_split,
        "batch pos head_idx head_dim, head_idx head_dim d_model -> batch pos head_idx d_model",
    )

    attn_out.save()

    logits = model.output.logits[:,-1,:]

    logits.save()
    value = logits.sum()
    value.backward()

with tl_model.trace(sample):
    tl_test = tl_model.blocks[8].attn.hook_result.output.save()

    tl_logits = tl_model.unembed.output[:,-1,:]

    tl_logits.save()

    value = tl_logits.sum()
    value.backward()

In [ ]:
test_resolution(tl_test, attn_out)

# TESTING MLP GRAD
- ok after visual inspection 

In [ ]:
sample = tokenizer.encode("Susan and Mary went to the store, Susan went in and")

with model.trace(sample):
    test = model.transformer.h[0].mlp.input[0][0].grad.save()

    logits = model.output.logits[:,-1,:]

    logits.save()
    value = logits.sum()
    value.backward()

with tl_model.trace(sample):
    tl_test = tl_model.blocks[0].mlp.input[0][0].grad.save()

    tl_logits = tl_model.unembed.output[:,-1,:]

    tl_logits.save()

    value = tl_logits.sum()
    value.backward()

In [ ]:
test

In [ ]:
tl_test

In [ ]:
test_resolution(tl_test, test)

# SPLIT Q INPUT

In [ ]:
sample = tokenizer.encode("Susan and Mary went to the store, Susan went in and")

slice_index = 0

with model.trace(sample):

    attention = model.transformer.h[0].attn
    attn_input = attention.input[0][0]


    repeated_attn = einops.repeat(
        attn_input,
        "batch pos d_model -> batch pos head_idx d_model",
        head_idx=12,
    ).clone()

    repeated_attn.grad.save()
    
    weight = torch.tensor_split(attention.c_attn.weight, 3, dim=1)[slice_index]
    split_weight = einops.rearrange(
        weight, 
        "d_model (head_index d_head) -> head_index d_model d_head",
        head_index=12
    )

    split_bias = einops.rearrange(
        attention.c_attn.bias,
        "(qkv head_idx d_head) -> qkv head_idx d_head",
        qkv=3,
        head_idx=12,
        d_head=64,
    )[slice_index]
    
    split_out = einops.einsum(
        repeated_attn, split_weight,
        "batch pos head_idx d_model, head_idx d_model d_head -> batch pos head_idx d_head",
    ) + split_bias
    
    split_attn = list(attention.c_attn.output.split(768, dim=2))

    split_attn[slice_index] = einops.rearrange(
       split_out, 
        "batch pos head_idx d_head -> batch pos (head_idx d_head)",
    )

    attention.c_attn.output = torch.cat(split_attn, dim=2)
    
    logits = model.output.logits[:,-1,:]

    value = logits.sum()
    value.backward()

# with tl_model.trace(sample):
#     tl_test = tl_model.blocks[0].mlp.input[0][0].grad.save()

#     tl_logits = tl_model.unembed.output[:,-1,:]

#     tl_logits.save()

#     value = tl_logits.sum()
#     value.backward()

In [ ]:
repeated_attn